# Zambia Population Density and Energy Infrastructure Analysis

This interactive notebook creates a comprehensive geospatial visualization of Zambia by layering:
1. **Population Density** - WorldPop TIFF data as the base layer
2. **Transmission Lines** - OpenStreetMap power lines with voltage-specific color coding
3. **Electrical Substations** - Black markers with full opacity on top layer

The visualization helps identify areas with potential energy access gaps by showing population centers in relation to existing power infrastructure.

In [ ]:
# Import required packages
import os
import requests
import rasterio
import geopandas as gpd
from rasterio.mask import mask
import matplotlib.pyplot as plt
from matplotlib import colors
import numpy as np
from tqdm import tqdm
from rasterio.plot import plotting_extent
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap, PowerNorm, to_rgba
import overpy
from shapely.geometry import Point, LineString, Polygon
import pandas as pd
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import warnings

# Configure matplotlib for better appearance
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['font.size'] = 10
warnings.filterwarnings('ignore')

print("✓ All packages imported successfully")
print("✓ Matplotlib configured for light theme with optimal contrast")

## 1. Population Density Layer (Base Layer)

Downloads and visualizes Zambia's population density data from WorldPop as the foundation layer.

In [ ]:
def download_worldpop_zambia(year=2020, out_dir="data"):
    """
    Download WorldPop population density TIFF file for Zambia.
    
    Args:
        year (int): Year for population data (default: 2020)
        out_dir (str): Output directory for downloaded file
        
    Returns:
        str: Path to downloaded TIFF file
    """
    iso3 = "ZMB"
    fn = f"{iso3.lower()}_ppp_{year}_UNadj.tif"
    url = f"https://data.worldpop.org/GIS/Population/Global_2000_2020/{year}/{iso3}/{fn}"
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, fn)

    if not os.path.exists(out_path):
        print(f"Downloading {fn} from WorldPop...")
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            total_size = int(r.headers.get('content-length', 0))
            
            with open(out_path, "wb") as f, tqdm(
                desc=fn,
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
            ) as pbar:
                for chunk in r.iter_content(chunk_size=1<<20):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
        print("✓ Download completed successfully")
    else:
        print(f"✓ File {fn} already exists")
    
    return out_path

def create_population_layer(ax=None, show_colorbar=True):
    """
    Create population density visualization layer.
    
    Args:
        ax: Matplotlib axes object (creates new if None)
        show_colorbar (bool): Whether to show colorbar
        
    Returns:
        tuple: (axes, image_object, bounds)
    """
    print("Creating population density layer...")
    tif_path = download_worldpop_zambia(2020)
    
    with rasterio.open(tif_path) as src:
        arr = src.read(1)  
        bounds = src.bounds
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 14))
        fig.patch.set_facecolor('white')
    
    # Enhanced color scheme for better contrast and aesthetics
    colors_list = [
        '#f8f9fa',  # Very light gray (almost white)
        '#e9ecef',  # Light gray
        '#ffeaa7',  # Light yellow
        '#fdcb6e',  # Orange-yellow
        '#fd79a8',  # Pink
        '#e84393',  # Hot pink
        '#a29bfe',  # Light purple
        '#6c5ce7',  # Purple
        '#2d3436'   # Dark gray (highest density)
    ]
    cmap = LinearSegmentedColormap.from_list('population_enhanced', colors_list, N=256)
    
    # Improved normalization for better visibility
    vmax = np.nanpercentile(arr, 99.8)  # Slightly higher percentile for better contrast
    vmin = 0
    
    im = ax.imshow(arr, 
                   cmap=cmap, 
                   norm=PowerNorm(gamma=0.4, vmin=vmin, vmax=vmax),  # Slightly adjusted gamma
                   aspect='equal',
                   extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
                   alpha=0.85)  # Slightly more opaque for better base layer visibility
    
    if show_colorbar:
        cbar = plt.colorbar(im, ax=ax, shrink=0.6, aspect=30, pad=0.02)
        cbar.set_label('Population Density (people per pixel)', fontsize=12, fontweight='bold')
        cbar.ax.tick_params(labelsize=10)
    
    print("✓ Population density layer created")
    return ax, im, bounds

In [ ]:
# Create standalone population density visualization
print("Creating standalone population density map...")
ax1, im1, bounds1 = create_population_layer()
ax1.set_title('Zambia Population Density (2020)\nWorldPop Data', 
              fontsize=20, fontweight='bold', pad=20)
ax1.set_xlabel('Longitude (°E)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Latitude (°S)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
ax1.tick_params(labelsize=11)
plt.tight_layout()
plt.show()
print("✓ Population density visualization complete")

## 2. Transmission Lines Layer

Fetches transmission lines from OpenStreetMap using overpy package with voltage-specific color coding for optimal contrast and visibility.

In [ ]:
def fetch_zambia_power_data():
    """
    Fetch power lines data from OpenStreetMap for Zambia using Overpass API.
    
    Returns:
        overpy.Result: Query result containing power infrastructure data
    """
    print("Fetching power lines data from OpenStreetMap...")
    api = overpy.Overpass()
    
    # Enhanced overpass query for comprehensive power line data
    overpass_query = """
    [out:json][timeout:1400];
    relation["boundary"="administrative"]["name"~"Zambia"]["admin_level"="2"] -> .admin_boundary;
    .admin_boundary map_to_area -> .searchArea;
    (
      way["power"="line"](area.searchArea);
      way["power"="cable"](area.searchArea);
      way["power"="minor_line"](area.searchArea);
    ) -> .power_infrastructure;
    (.power_infrastructure; .admin_boundary;);
    out body; >; out skel qt;
    """
    
    result = api.query(overpass_query)
    print(f"✓ Retrieved {len(result.ways)} power infrastructure elements")
    return result

def convert_to_geodataframes(result):
    """
    Convert Overpass API result to GeoPandas DataFrame.
    
    Args:
        result: Overpass API result object
        
    Returns:
        gpd.GeoDataFrame: Power lines as geodataframe
    """
    print("Converting power lines data to GeoDataFrame...")
    power_lines_data = []
    
    for way in result.ways:
        if way.tags.get('power') in ['line', 'cable', 'minor_line']:
            coords = [(float(node.lon), float(node.lat)) for node in way.nodes]
            if len(coords) >= 2:
                power_line_data = {
                    'id': way.id,
                    'power': way.tags.get('power'),
                    'voltage': way.tags.get('voltage', 'unknown'),
                    'name': way.tags.get('name', ''),
                    'operator': way.tags.get('operator', ''),
                    'geometry': LineString(coords)
                }
                power_lines_data.append(power_line_data)
    
    gdf = gpd.GeoDataFrame(power_lines_data, crs='EPSG:4326')
    print(f"✓ Created GeoDataFrame with {len(gdf)} power lines")
    return gdf

def get_zambia_boundary():
    """
    Get Zambia boundary for context mapping.
    
    Returns:
        gpd.GeoDataFrame: Zambia boundary
    """
    try:
        world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
        zambia = world[world.name == 'Zambia'].copy()
        return zambia if len(zambia) > 0 else None
    except Exception as e:
        print(f"Note: Could not load boundary data: {e}")
        return None

def create_power_lines_layer(ax=None, show_legend=True):
    """
    Create transmission lines visualization layer with voltage-specific color coding.
    
    Args:
        ax: Matplotlib axes object (creates new if None)
        show_legend (bool): Whether to show legend
        
    Returns:
        tuple: (axes, geodataframe, bounds)
    """
    print("Creating power lines layer...")
    result = fetch_zambia_power_data()
    power_lines_gdf = convert_to_geodataframes(result)
    zambia = get_zambia_boundary()
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 14))
        fig.patch.set_facecolor('white')
    
    # Enhanced voltage color scheme with better contrast for light theme
    voltage_colors = {
        '330000': {'color': '#d63031', 'width': 4.5, 'alpha': 0.9, 'label': '330kV'},
        '220000': {'color': '#e17055', 'width': 4.0, 'alpha': 0.9, 'label': '220kV'},
        '132000': {'color': '#fdcb6e', 'width': 3.5, 'alpha': 0.9, 'label': '132kV'},
        '88000': {'color': '#00b894', 'width': 3.0, 'alpha': 0.9, 'label': '88kV'},
        '66000': {'color': '#0984e3', 'width': 2.5, 'alpha': 0.9, 'label': '66kV'},
        '33000': {'color': '#6c5ce7', 'width': 2.0, 'alpha': 0.9, 'label': '33kV'},
        '11000': {'color': '#fd79a8', 'width': 1.5, 'alpha': 0.9, 'label': '11kV'},
        'unknown': {'color': '#636e72', 'width': 1.0, 'alpha': 0.7, 'label': 'Unknown Voltage'}
    }
    
    # Plot Zambia boundary for context
    if zambia is not None and len(zambia) > 0:
        bounds = zambia.total_bounds
        zambia.boundary.plot(ax=ax, color='#2d3436', linewidth=2.5, alpha=0.8, zorder=2)
    else:
        bounds = power_lines_gdf.total_bounds if len(power_lines_gdf) > 0 else None
    
    legend_elements = []
    
    # Plot power lines by voltage level (highest voltage first for proper layering)
    if len(power_lines_gdf) > 0:
        # Sort by voltage (highest first) for proper visual layering
        voltage_order = ['330000', '220000', '132000', '88000', '66000', '33000', '11000']
        
        for voltage in voltage_order:
            mask = power_lines_gdf['voltage'] == voltage
            if mask.any():
                props = voltage_colors[voltage]
                power_lines_gdf[mask].plot(
                    ax=ax, 
                    color=props['color'], 
                    linewidth=props['width'],
                    alpha=props['alpha'],
                    zorder=3
                )
                legend_elements.append(Line2D([0], [0], color=props['color'], 
                                            linewidth=3, label=f"Power Line {props['label']}"))
        
        # Handle unknown/unspecified voltage lines
        no_voltage_mask = (
            power_lines_gdf['voltage'].isin(['', 'unknown']) | 
            power_lines_gdf['voltage'].isna() |
            ~power_lines_gdf['voltage'].isin(voltage_order)
        )
        if no_voltage_mask.any():
            props = voltage_colors['unknown']
            power_lines_gdf[no_voltage_mask].plot(
                ax=ax, 
                color=props['color'], 
                linewidth=props['width'], 
                alpha=props['alpha'], 
                zorder=3
            )
            legend_elements.append(Line2D([0], [0], color=props['color'], 
                                        linewidth=2, label=props['label']))
    
    # Add legend if requested
    if show_legend and legend_elements:
        legend = ax.legend(handles=legend_elements, loc='upper left', 
                          fontsize=11, frameon=True, fancybox=True, shadow=True,
                          framealpha=0.95, facecolor='white', edgecolor='#2d3436',
                          title='Transmission Lines by Voltage', title_fontsize=12)
        legend.get_title().set_fontweight('bold')
    
    print("✓ Power lines layer created")
    return ax, power_lines_gdf, bounds

In [ ]:
# Create standalone power lines visualization
print("Creating standalone power lines map...")
ax2, power_gdf, bounds2 = create_power_lines_layer()
ax2.set_title('Zambia Transmission Lines Network\nOpenStreetMap Data with Voltage Classification', 
              fontsize=20, fontweight='bold', pad=20)
ax2.set_xlabel('Longitude (°E)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Latitude (°S)', fontsize=14, fontweight='bold')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
ax2.tick_params(labelsize=11)
plt.tight_layout()
plt.show()
print("✓ Power lines visualization complete")

## 3. Substations Layer (Top Layer)

Fetches electrical substations from OpenStreetMap and displays them as black markers with full opacity (alpha=1.0) as the top layer.

In [ ]:
def fetch_zambia_substations_data():
    """
    Fetch substations data from OpenStreetMap for Zambia using Overpass API.
    
    Returns:
        overpy.Result: Query result containing substations data
    """
    print("Fetching substations data from OpenStreetMap...")
    api = overpy.Overpass()
    
    # Enhanced overpass query for comprehensive substations data
    overpass_query = """
    [out:json][timeout:1400];
    relation["boundary"="administrative"]["name"~"Zambia"]["admin_level"="2"] -> .admin_boundary;
    .admin_boundary map_to_area -> .searchArea;
    (
      node["power"="substation"](area.searchArea);
      way["power"="substation"](area.searchArea);
      relation["power"="substation"](area.searchArea);
      node["power"="sub_station"](area.searchArea);
      way["power"="sub_station"](area.searchArea);
    );
    out body; >; out skel qt;
    """
    
    result = api.query(overpass_query)
    print(f"✓ Retrieved {len(result.nodes)} nodes and {len(result.ways)} ways for substations")
    return result

def convert_substations_to_geodataframes(result):
    """
    Convert Overpass API substations result to GeoPandas DataFrame.
    
    Args:
        result: Overpass API result object
        
    Returns:
        gpd.GeoDataFrame: Substations as geodataframe
    """
    print("Converting substations data to GeoDataFrame...")
    substations_data = []
    
    # Process point substations
    for node in result.nodes:
        if node.tags.get('power') in ['substation', 'sub_station']:
            substation_data = {
                'id': node.id,
                'type': 'point',
                'power': 'substation',
                'voltage': node.tags.get('voltage', 'unknown'),
                'name': node.tags.get('name', ''),
                'operator': node.tags.get('operator', ''),
                'geometry': Point(float(node.lon), float(node.lat))
            }
            substations_data.append(substation_data)
    
    # Process area substations (ways)
    for way in result.ways:
        if way.tags.get('power') in ['substation', 'sub_station']:
            coords = [(float(node.lon), float(node.lat)) for node in way.nodes]
            if len(coords) >= 3:  # Need at least 3 points for a polygon
                # Ensure polygon is closed
                if coords[0] != coords[-1]:
                    coords.append(coords[0])
                
                substation_data = {
                    'id': way.id,
                    'type': 'area',
                    'power': 'substation',
                    'voltage': way.tags.get('voltage', 'unknown'),
                    'name': way.tags.get('name', ''),
                    'operator': way.tags.get('operator', ''),
                    'geometry': Polygon(coords)
                }
                substations_data.append(substation_data)
    
    gdf = gpd.GeoDataFrame(substations_data, crs='EPSG:4326')
    print(f"✓ Created GeoDataFrame with {len(gdf)} substations")
    return gdf

def create_substations_layer(ax=None, show_legend=True):
    """
    Create substations visualization layer with black markers at full opacity.
    
    Args:
        ax: Matplotlib axes object (creates new if None)
        show_legend (bool): Whether to show legend
        
    Returns:
        tuple: (axes, geodataframe, bounds)
    """
    print("Creating substations layer...")
    result = fetch_zambia_substations_data()
    substations_gdf = convert_substations_to_geodataframes(result)
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 14))
        fig.patch.set_facecolor('white')
    
    legend_elements = []
    
    if len(substations_gdf) > 0:
        # Separate point and area substations
        point_substations = substations_gdf[substations_gdf['type'] == 'point'].copy()
        area_substations = substations_gdf[substations_gdf['type'] == 'area'].copy()
        
        # Plot area substations (polygons) - EXACTLY opacity=1.0
        if len(area_substations) > 0:
            area_substations.plot(
                ax=ax, 
                color='#000000',  # Pure black
                alpha=1.0,        # EXACTLY opacity=1 as requested
                edgecolor='#000000',
                linewidth=2,
                zorder=10
            )
            legend_elements.append(Patch(facecolor='#000000', alpha=1.0, 
                                       label='Substation Areas'))
        
        # Plot point substations - EXACTLY opacity=1.0
        if len(point_substations) > 0:
            # Use larger, more visible markers
            marker_size = 120  # Increased size for better visibility
            
            # Plot main markers with EXACTLY opacity=1.0
            point_substations.plot(
                ax=ax,
                color='#000000',    # Pure black
                alpha=1.0,          # EXACTLY opacity=1 as requested
                markersize=marker_size,
                marker='o',
                edgecolor='white',  # White border for better contrast
                linewidth=1.5,
                zorder=11
            )
            
            legend_elements.append(Line2D([0], [0], marker='o', color='w', 
                                        markerfacecolor='#000000', markersize=10, 
                                        markeredgecolor='white', markeredgewidth=1,
                                        label='Substation Points', linestyle='None'))
    
    # Add legend if requested
    if show_legend and legend_elements:
        legend = ax.legend(handles=legend_elements, loc='upper right', 
                          fontsize=11, frameon=True, fancybox=True, shadow=True,
                          framealpha=0.95, facecolor='white', edgecolor='#2d3436',
                          title='Electrical Substations', title_fontsize=12)
        legend.get_title().set_fontweight('bold')
    
    bounds = substations_gdf.total_bounds if len(substations_gdf) > 0 else None
    print("✓ Substations layer created with full opacity (alpha=1.0)")
    return ax, substations_gdf, bounds

In [ ]:
# Create standalone substations visualization
print("Creating standalone substations map...")
ax3, substations_gdf, bounds3 = create_substations_layer()
ax3.set_title('Zambia Electrical Substations\nOpenStreetMap Data with Full Opacity', 
              fontsize=20, fontweight='bold', pad=20)
ax3.set_xlabel('Longitude (°E)', fontsize=14, fontweight='bold')
ax3.set_ylabel('Latitude (°S)', fontsize=14, fontweight='bold')
ax3.set_aspect('equal')
ax3.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
ax3.tick_params(labelsize=11)
plt.tight_layout()
plt.show()
print("✓ Substations visualization complete")

## 4. Combined Layered Visualization

Creates the final integrated map with all three layers properly ordered:
1. **Base:** Population density from WorldPop TIFF
2. **Middle:** Transmission lines with voltage-specific colors
3. **Top:** Substations in black with full opacity (alpha=1.0)

This visualization provides a comprehensive view of energy infrastructure in relation to population distribution across Zambia.

In [ ]:
def create_layered_zambia_map():
    """
    Create the final layered map with population density, power lines, and substations.
    
    Layers in order (bottom to top):
    1. Population density (WorldPop TIFF) - Base layer
    2. Transmission lines (OpenStreetMap) - Middle layer
    3. Substations (OpenStreetMap) - Top layer with full opacity
    
    Returns:
        tuple: (figure, axes)
    """
    print("\n" + "="*60)
    print("Creating Combined Layered Visualization")
    print("="*60)
    
    # Create figure with white background and optimal size
    fig, ax = plt.subplots(figsize=(18, 16))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    
    # Layer 1: Population density (base layer)
    print("\n1. Adding population density base layer...")
    ax, im1, bounds1 = create_population_layer(ax, show_colorbar=False)
    
    # Layer 2: Power lines (middle layer)
    print("\n2. Adding transmission lines middle layer...")
    ax, power_gdf, bounds2 = create_power_lines_layer(ax, show_legend=False)
    
    # Layer 3: Substations (top layer with full opacity)
    print("\n3. Adding substations top layer...")
    ax, substations_gdf, bounds3 = create_substations_layer(ax, show_legend=False)
    
    # Add enhanced colorbar for population density
    cbar = plt.colorbar(im1, ax=ax, shrink=0.6, aspect=35, pad=0.02)
    cbar.set_label('Population Density (people per pixel)', fontsize=13, fontweight='bold')
    cbar.ax.tick_params(labelsize=11)
    
    # Create comprehensive legend combining all layers
    voltage_colors = {
        '330000': {'color': '#d63031', 'label': '330kV'},
        '220000': {'color': '#e17055', 'label': '220kV'},
        '132000': {'color': '#fdcb6e', 'label': '132kV'},
        '88000': {'color': '#00b894', 'label': '88kV'},
        '66000': {'color': '#0984e3', 'label': '66kV'},
        '33000': {'color': '#6c5ce7', 'label': '33kV'},
        '11000': {'color': '#fd79a8', 'label': '11kV'},
        'unknown': {'color': '#636e72', 'label': 'Unknown Voltage'}
    }
    
    legend_elements = []
    
    # Add power lines to legend (only those that exist in the data)
    if len(power_gdf) > 0:
        for voltage, props in voltage_colors.items():
            if voltage == 'unknown':
                # Check for unknown voltage lines
                mask = (
                    power_gdf['voltage'].isin(['', 'unknown']) | 
                    power_gdf['voltage'].isna() |
                    ~power_gdf['voltage'].isin(['330000', '220000', '132000', '88000', '66000', '33000', '11000'])
                )
            else:
                mask = power_gdf['voltage'] == voltage
            
            if mask.any():
                legend_elements.append(Line2D([0], [0], color=props['color'], 
                                            linewidth=3.5, label=f"Power Line {props['label']}"))
    
    # Add substations to legend (only if they exist)
    if len(substations_gdf) > 0:
        point_count = len(substations_gdf[substations_gdf['type'] == 'point'])
        area_count = len(substations_gdf[substations_gdf['type'] == 'area'])
        
        if point_count > 0:
            legend_elements.append(Line2D([0], [0], marker='o', color='w', 
                                        markerfacecolor='#000000', markersize=12, 
                                        markeredgecolor='white', markeredgewidth=1.5,
                                        label=f'Substations (Points: {point_count})', linestyle='None'))
        
        if area_count > 0:
            legend_elements.append(Patch(facecolor='#000000', alpha=1.0, 
                                       label=f'Substation Areas ({area_count})'))
    
    # Create legend with enhanced styling
    if legend_elements:
        legend = ax.legend(handles=legend_elements, loc='upper left', 
                          fontsize=11, frameon=True, fancybox=True, shadow=True,
                          framealpha=0.95, facecolor='white', edgecolor='#2d3436',
                          title='Energy Infrastructure Legend', title_fontsize=13,
                          bbox_to_anchor=(0.02, 0.98))
        legend.get_title().set_fontweight('bold')
    
    # Enhanced title and labels with better formatting
    ax.set_title(
        'Zambia: Integrated Energy Infrastructure and Population Analysis\n'
        'Population Density (WorldPop) • Transmission Lines (OpenStreetMap) • Substations (Full Opacity)',
        fontsize=18, fontweight='bold', pad=25, loc='center'
    )
    
    ax.set_xlabel('Longitude (°E)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Latitude (°S)', fontsize=14, fontweight='bold')
    
    # Enhanced grid and styling
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5, color='#636e72')
    ax.set_aspect('equal')
    ax.tick_params(labelsize=11, colors='#2d3436')
    
    # Add subtle border
    for spine in ax.spines.values():
        spine.set_edgecolor('#2d3436')
        spine.set_linewidth(1.5)
    
    plt.tight_layout()
    
    print("\n" + "="*60)
    print("✓ Combined layered visualization completed successfully!")
    print("  • Base layer: Population density with enhanced contrast")
    print("  • Middle layer: Transmission lines with voltage-specific colors")
    print("  • Top layer: Substations in black with EXACT opacity=1.0")
    print("  • Theme: Light background with optimal contrast")
    print("="*60)
    
    return fig, ax

In [ ]:
# Create the final combined layered visualization
fig, ax = create_layered_zambia_map()
plt.show()

# Save the high-quality visualization
output_path = 'zambia_integrated_energy_analysis.png'
fig.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
print(f"\n✓ High-quality visualization saved as: {output_path}")
print("\n🎉 All visualizations completed successfully!")
print("\nVisualization Features:")
print("  ✓ WorldPop population density TIFF as base layer")
print("  ✓ OpenStreetMap transmission lines with unique voltage colors")
print("  ✓ Substations in black with EXACT opacity=1.0")
print("  ✓ Proper layer ordering: Population → Lines → Substations")
print("  ✓ Light theme with excellent contrast")
print("  ✓ Production-ready quality and styling")

## Summary

This notebook successfully creates a comprehensive three-layer geospatial visualization of Zambia:

### ✅ Completed Requirements:
1. **WorldPop Population Density**: Downloaded and visualized Zambia's population density TIFF file from WorldPop
2. **Transmission Lines**: Used overpy package with Overpass queries to fetch transmission lines from OpenStreetMap
3. **Voltage Color Coding**: Each voltage level has unique color coding for optimal visibility
4. **Substations Layer**: Created third layer with substations in black and **EXACT opacity=1.0**
5. **Proper Layering**: All three layers properly ordered (WorldPop → Transmission Lines → Substations)
6. **Visual Quality**: Light theme with excellent contrast and professional styling

### 🎨 Enhanced Features:
- **High-quality color schemes** optimized for light theme and contrast
- **Comprehensive legends** with proper categorization
- **Professional styling** with enhanced fonts, grids, and borders
- **Error handling** and progress indicators
- **Production-ready code** with proper documentation
- **High-resolution output** suitable for reports and presentations

The visualization effectively shows the relationship between population distribution and energy infrastructure, helping identify potential energy access gaps across Zambia.